# Rodada 30 — gate convolucional S12 x Analog (treino no Kaggle)

Notebook de infraestrutura: monta a mesma estrutura de pastas do
repositório local dentro de `/kaggle/working/repo`, para que
`src/round30_cnn.py` (e os módulos que ele importa) rodem sem
modificação, exatamente como rodam localmente via `python -m src.roundNN`.

Antes de rodar: confira em **Add Input** que os dois datasets abaixo
estão anexados:
1. O dataset oficial da competição WORCAP 2026 (dados brutos).
2. `worcap-round30-artefatos` (privado, gerado por
   `scripts/package_round30_kaggle.py` + `upload_round30_kaggle.py`).

Confira o Accelerator em **Settings -> GPU T4 x2** (ou P100) antes de
rodar a célula de treino.

## 1. Preencher os dois caminhos abaixo

In [ ]:
# EDITE ESTAS DUAS LINHAS conforme o painel "Add Input" do notebook.
ARTIFACTS_INPUT = "/kaggle/input/worcap-round30-artefatos"
COMPETITION_INPUT = "/kaggle/input/NOME-DO-DATASET-OFICIAL-AQUI"

import os
print("artefatos:", os.listdir(ARTIFACTS_INPUT))
print("dados oficiais:", os.listdir(COMPETITION_INPUT))


## 2. Montar o layout do repositório em `/kaggle/working/repo`

`ROOT` dentro de `src/competition.py` é calculado como
`Path(__file__).resolve().parents[1]`. Reproduzindo a mesma estrutura
relativa (`repo/src/...`, `repo/data/processed/...`, `repo/data/raw/...`),
todo o código existente funciona sem precisar reescrever caminhos.

In [ ]:
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
(REPO / "src").mkdir(parents=True)
(REPO / "data" / "raw").mkdir(parents=True)
(REPO / "experiments").mkdir(parents=True)

# Código: todos os módulos empacotados em ARTIFACTS_INPUT/code
for py in Path(ARTIFACTS_INPUT, "code").glob("*.py"):
    shutil.copy2(py, REPO / "src" / py.name)

# Protocolo da rodada 30 (só o hash importa para `locked()`)
protocol_src = Path(ARTIFACTS_INPUT, "code", "ROUND30.md")
if protocol_src.exists():
    shutil.copy2(protocol_src, REPO / "experiments" / "ROUND30.md")
else:
    print("AVISO: ROUND30.md nao veio no bundle; copie manualmente para",
          REPO / "experiments" / "ROUND30.md")

# Artefatos congelados: replica a mesma árvore data/processed/*
dest_processed = REPO / "data" / "processed"
for sub in ("round27", "round28"):
    shutil.copytree(Path(ARTIFACTS_INPUT, "artifacts", sub), dest_processed / sub)
shutil.copytree(Path(ARTIFACTS_INPUT, "artifacts", "s12_evidence"),
                REPO / "delivery" / "s12" / "evidence", dirs_exist_ok=True)
for sub in ("round9", "round10", "round20", "round25"):
    src_dir = Path(ARTIFACTS_INPUT, "artifacts", sub)
    if src_dir.exists():
        shutil.copytree(src_dir, dest_processed / sub, dirs_exist_ok=True)

print("Repositório montado em", REPO)


## 3. Ligar `data/raw` aos dados oficiais anexados

Ajuste os nomes de arquivo abaixo caso o dataset oficial do Kaggle use
nomes diferentes dos locais (`treino_tp.nc`, `treino_tp_alvo.nc`,
`treino_<variavel>.nc` para as 9 variáveis, `teste_features.nc`,
`sample_submission.csv`). `src/competition.py` espera exatamente esses
nomes em `data/raw/`.

In [ ]:
import os

EXPECTED = [
    "treino_tp.nc", "treino_tp_alvo.nc", "teste_features.nc", "sample_submission.csv",
    "treino_t2.nc", "treino_cloud_cover.nc", "treino_shum_850.nc",
    "treino_surface_pressure.nc", "treino_u_850.nc", "treino_v_850.nc",
    "treino_temperature_850.nc", "treino_rel_hum_850.nc", "treino_geopotential_850.nc",
]
raw_dir = REPO / "data" / "raw"
missing = []
for name in EXPECTED:
    source = Path(COMPETITION_INPUT, name)
    target = raw_dir / name
    if source.exists():
        if not target.exists():
            os.symlink(source, target)
    else:
        missing.append(name)

if missing:
    print("NAO ENCONTRADOS em COMPETITION_INPUT (confira o nome exato no painel Add Input):")
    for name in missing:
        print(" -", name)
else:
    print("Todos os arquivos oficiais linkados em", raw_dir)


## 4. Reconstruir o cache oficial (`tp.npy` e as 9 variáveis)

Reusa `competition.audit()` sem modificação -- a mesma função que roda
localmente. Isso recria `data/processed/official/tp.npy`, necessário
para `round30_cnn.evaluate()`. Não redistribuímos esse arquivo pelo
dataset privado; ele é sempre reconstruído a partir do dataset oficial
anexado.

In [ ]:
# O Kaggle às vezes não registra os metadados de pacote do netCDF4 do
# jeito que importlib.metadata espera, mesmo com a leitura de NetCDF
# funcionando. Isso evita um PackageNotFoundError cosmético dentro de
# competition.audit() (a linha que só lista versões de bibliotecas).
%pip install -q netCDF4

import sys
sys.path.insert(0, str(REPO))

from src import competition
print("audit() vai gravar o cache em:", competition.CACHE)
competition.audit()


## 5. Treinar e avaliar o gate convolucional (Rodada 30)

Isso executa `evaluate()` (treina um `GateCNN` por corte causal e salva o
OOF em `data/processed/round30/`) e depois `decision()` (aplica a mesma
aritmética de soft gating da Rodada 27 e classifica A/B/D). Acompanhe a
cota de GPU -- salve o notebook periodicamente; `evaluate()` pula blocos
já salvos se a sessão cair e for reiniciada.

In [ ]:
import torch
print("GPU disponivel:", torch.cuda.is_available(), torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "(nenhuma)")

from src import round30_cnn
round30_cnn.evaluate()
result = round30_cnn.decision()
result


## 6. Salvar para trazer de volta ao repositório

Copia os JSON/NPZ de `reports/competition/round30` e
`data/processed/round30` para `/kaggle/working/round30_output/`, que
aparece na aba **Output** do notebook para download, ou pode ser subido
como nova versão do dataset `worcap-round30-artefatos` a partir daqui
mesmo (célula opcional comentada).

In [ ]:
import shutil

OUT_DIR = Path("/kaggle/working/round30_output")
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
shutil.copytree(REPO / "reports" / "competition" / "round30", OUT_DIR / "reports", dirs_exist_ok=True)
shutil.copytree(REPO / "data" / "processed" / "round30", OUT_DIR / "oof", dirs_exist_ok=True)
print("Pronto para baixar em", OUT_DIR)

# Opcional: subir como nova versao do dataset privado direto daqui.
# import kagglehub
# kagglehub.login()
# kagglehub.dataset_upload("SEU_USUARIO_KAGGLE/worcap-round30-artefatos", str(OUT_DIR),
#                           version_notes="rodada 30: resultados do gate CNN")
